# Stage 3 — Pipeline Evaluation & Validation

Val set'te top-K grid search + threshold calibration yapılır,  
ardından aynı threshold test setine uygulanarak final metrikler hesaplanır.

**Outputs:** `artifacts/threshold.json`, `outputs/val_*`, `outputs/test_*`

In [ ]:
import os, sys, json
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score,
    roc_curve, balanced_accuracy_score
)

PROJECT_ROOT = "/home/yasemin/video-anomaly-clip"
sys.path.append(PROJECT_ROOT)

from src.inference import CLIPInference
from src.aggregation import aggregate_video_scores
from src.config import ANOMALY_CLASSES, NORMAL_CLASSES

## Config

In [ ]:
DATA_ROOT     = Path(PROJECT_ROOT) / "data"
MANIFEST_PATH = DATA_ROOT / "manifests_new" / "manifest.csv"
SEGMENTS_ROOT = DATA_ROOT / "segments_new"

# Hangi veri kümesi / kurulum — artifacts/threshold.json'a yazılır
DATASET_MODE = "ucf_crime"

# Initial placeholder — grid search (top-K cell) tarafından üzerine yazılır
TOPK_RATIO  = 0.10
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

os.makedirs(f"{PROJECT_ROOT}/artifacts", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/outputs",   exist_ok=True)

print("Config OK")

## Load Manifest

In [ ]:
df = pd.read_csv(MANIFEST_PATH)

# Manifest was generated on Colab; remap segment paths to local filesystem
df["path"] = df["path"].str.replace(
    "/content/drive/MyDrive/clip_delivery_new/segments",
    str(SEGMENTS_ROOT),
    regex=False
)

print(f"Manifest: {len(df)} rows")
print(df.columns.tolist())
print(df.head(3))
print("\nSplit distribution:")
print(df.groupby(["split", "label"]).size())

# Sanity check: paths must exist on disk
for p in df["path"].head(3):
    print(f"  {p} -> {Path(p).exists()}")

## Val / Test Split

In [ ]:
val_df  = df[df["split"] == "val"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

print(f"Val  : {len(val_df)} segment")
print(f"Test : {len(test_df)} segment")

## Model Init

In [ ]:
clip_model = CLIPInference()
clip_model.set_text_prompts()
print("CLIP model initialized")

## Segment-Level Inference (Val)

In [ ]:
results = []
errors  = []

print("\nRunning segment-level inference (val)...\n")

for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
    full_path  = Path(row["path"])
    true_class = row["class"]
    video_id   = row["video_id"]

    try:
        pred = clip_model.predict_segment(str(full_path))
        results.append({
            "video_id"    : video_id,
            "segment_path": str(full_path),
            "true_class"  : true_class,
            "label"       : row["label"],
            "anomaly_sim" : pred["anomaly_sim"],
            "normal_sim"  : pred["normal_sim"],
            "score"       : pred["score"],
        })
    except Exception as e:
        errors.append({"segment": str(full_path), "error": repr(e)})
        print(f"\nERROR: {full_path} -> {repr(e)}")

results_df = pd.DataFrame(results)
print(f"\nInference done. Success: {len(results_df)} | Errors: {len(errors)}")
print(results_df["true_class"].value_counts())

## Video-Level Aggregation

In [ ]:
video_df = aggregate_video_scores(results_df)
print(f"Videos: {len(video_df)}")
video_df.head()

## Top-K Grid Search

In [ ]:
# =====================================================================
# Top-K Grid Search + RECALL-ONCELIKLI Threshold Calibration  [VAL]
# =====================================================================
# Bu sistem bir ON-ELEME aracidir: amac anomalileri YAKALAMAK (yuksek recall).
# Normalleri reddetmek (specificity) ikincil. Bu yuzden esigi balanced accuracy
# yerine "val recall >= hedef olan noktalar icinde EN IYI specificity" ile seceriz.
# Recall boylece feda edilmeyen sert bir kisit olur.
# Anomali zamanda seyrek oldugu icin kucuk K (max'a yakin) anomalik segmenti
# one cikarir; grid search uygun orani recall hedefine gore kendisi secer.
#
# NOT: RATIOS, aggregation.py icindeki TOPK_RATIOS ile BIREBIR AYNI olmali.

RATIOS = [0.05, 0.10, 0.15, 0.20, 0.30, 0.50, 1.00]
TARGET_RECALL = 0.80   # <-- AYAR DUGMESI: val'de yakalanmasi gereken min anomali orani

y_true = video_df["is_anomaly"].astype(int).values

# --- Manzara: her oranin ulasabilecegi en iyi recall ve hedefte specificity ---
print(f"{'ratio':>6} | {'max_recall':>10} | {'spec@rec>=' + format(TARGET_RECALL,'.2f'):>14} | {'best_BA':>7}")
print("-" * 48)

best     = None   # (spec, ratio, threshold, recall) : hedefi tutturanlar icinde en iyi spec
fallback = None   # (spec, ratio, threshold, recall) : en yuksek recall (hedef ulasilamazsa)

for ratio in RATIOS:
    a_col, n_col = f"topk_a_{ratio}", f"topk_n_{ratio}"
    if a_col not in video_df.columns:
        raise KeyError(f"'{a_col}' yok. aggregation.py guncel mi / RATIOS==TOPK_RATIOS mi?")
    sd = (video_df[a_col] - video_df[n_col]).values

    r_maxrec, r_spec_at, r_bestba = -1.0, None, 0.0
    for t in np.linspace(sd.min(), sd.max(), 500):
        yp = (sd > t).astype(int)
        if len(np.unique(yp)) < 2:
            continue
        tn, fp, fn, tp = confusion_matrix(y_true, yp).ravel()
        rec  = tp / (tp + fn + 1e-8)
        spec = tn / (tn + fp + 1e-8)
        r_maxrec = max(r_maxrec, rec)
        r_bestba = max(r_bestba, (rec + spec) / 2)
        if rec >= TARGET_RECALL:
            if r_spec_at is None or spec > r_spec_at:
                r_spec_at = spec
            if best is None or spec > best[0]:
                best = (spec, ratio, t, rec)
        if fallback is None or rec > fallback[3]:
            fallback = (spec, ratio, t, rec)

    shown = f"{r_spec_at:.3f}" if r_spec_at is not None else "  -  "
    print(f"{ratio:>6} | {r_maxrec:>10.3f} | {shown:>14} | {r_bestba:>7.3f}")

if best is None:
    print(f"\nUYARI: val'de recall>={TARGET_RECALL} hicbir oranda ulasilamadi -> en yuksek recall noktasi secildi.")
    best = fallback

best_spec, TOPK_RATIO, best_threshold, _ = best

# --- Secilen (oran, esik) ile nihai val skoru ---
video_df["score_diff"] = (
    video_df[f"topk_a_{TOPK_RATIO}"] - video_df[f"topk_n_{TOPK_RATIO}"]
)
scores = video_df["score_diff"].values
video_df["is_anomaly"]           = y_true.astype(bool)
video_df["predicted_is_anomaly"] = (scores > best_threshold)

tn, fp, fn, tp = confusion_matrix(y_true, (scores > best_threshold).astype(int)).ravel()
recall      = tp / (tp + fn + 1e-8)
specificity = tn / (tn + fp + 1e-8)

print("\n" + "=" * 44)
print(f"Secilen oran (TOPK_RATIO) : {TOPK_RATIO}")
print(f"Secilen esik (threshold)  : {best_threshold:.6f}")
print(f"Recall (val)              : {recall:.4f}   <- hedef {TARGET_RECALL}")
print(f"Specificity (val)         : {specificity:.4f}")
print(f"Balanced Accuracy (val)   : {(recall + specificity) / 2:.4f}")
print(f"TP={tp}  TN={tn}  FP={fp}  FN={fn}")

## Validation Metrics

In [ ]:
y_true_bool = video_df["is_anomaly"].astype(bool)
y_pred_bool = video_df["predicted_is_anomaly"].astype(bool)
scores      = video_df["score_diff"]

accuracy     = accuracy_score(y_true_bool, y_pred_bool)
precision    = precision_score(y_true_bool, y_pred_bool, zero_division=0)
recall       = recall_score(y_true_bool, y_pred_bool, zero_division=0)
f1           = f1_score(y_true_bool, y_pred_bool, zero_division=0)
roc_auc      = roc_auc_score(y_true_bool, scores)
tn, fp, fn, tp = confusion_matrix(y_true_bool, y_pred_bool).ravel()
specificity  = tn / (tn + fp + 1e-8)
balanced_acc = (recall + specificity) / 2

print("=" * 50)
print("VALIDATION METRICS")
print("=" * 50)
print(f"Accuracy         : {accuracy:.4f}")
print(f"Balanced Accuracy: {balanced_acc:.4f}")
print(f"Precision        : {precision:.4f}")
print(f"Recall           : {recall:.4f}")
print(f"Specificity      : {specificity:.4f}")
print(f"F1               : {f1:.4f}")
print(f"ROC-AUC          : {roc_auc:.4f}")
print(f"TP={tp}  TN={tn}  FP={fp}  FN={fn}")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true_bool, y_pred_bool)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Anomaly'],
            yticklabels=['Normal', 'Anomaly'])
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion Matrix (Validation)")
plt.tight_layout()
plt.savefig(f"{PROJECT_ROOT}/outputs/val_confusion_matrix.png", dpi=150)
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_true_bool, scores)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC={roc_auc:.4f}")
plt.plot([0, 1], [0, 1], "k--", alpha=.4)
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title("ROC Curve (Validation)"); plt.legend()
plt.tight_layout()
plt.savefig(f"{PROJECT_ROOT}/outputs/val_roc_curve.png", dpi=150)
plt.show()

# Score Distribution
plt.figure(figsize=(8, 5))
plt.hist(video_df[~video_df["is_anomaly"]]["score_diff"], bins=15, alpha=0.6, label="Normal")
plt.hist(video_df[ video_df["is_anomaly"]]["score_diff"], bins=15, alpha=0.6, label="Anomaly")
plt.axvline(best_threshold, linestyle="--", color="red", label=f"thr={best_threshold:.4f}")
plt.xlabel("Score Diff"); plt.ylabel("Count")
plt.title("Score Distribution (Validation)"); plt.legend()
plt.tight_layout()
plt.savefig(f"{PROJECT_ROOT}/outputs/val_score_dist.png", dpi=150)
plt.show()

print("\nPer-video scores:")
print(video_df[["video_id", "true_class", "score_diff", "predicted_is_anomaly"]].to_string())

## Save Val Artifacts

In [ ]:
with open(f"{PROJECT_ROOT}/artifacts/threshold.json", "w") as f:
    json.dump({
        "dataset_mode": DATASET_MODE,
        "threshold"   : float(best_threshold),
        "topk_ratio"  : TOPK_RATIO,
        "metrics": {
            "accuracy"        : float(accuracy),
            "balanced_accuracy": float(balanced_acc),
            "precision"       : float(precision),
            "recall"          : float(recall),
            "specificity"     : float(specificity),
            "f1"              : float(f1),
            "roc_auc"         : float(roc_auc),
            "tp": int(tp), "tn": int(tn),
            "fp": int(fp), "fn": int(fn),
        }
    }, f, indent=4)

video_df.to_csv(f"{PROJECT_ROOT}/outputs/val_video_predictions.csv", index=False)
results_df.to_csv(f"{PROJECT_ROOT}/outputs/val_segment_predictions.csv", index=False)

print("✅  Artifacts saved:")
print(f"    {PROJECT_ROOT}/artifacts/threshold.json")
print(f"    {PROJECT_ROOT}/outputs/val_video_predictions.csv")

## Segment-Level Inference (Test)

In [ ]:
test_results = []
test_errors  = []

print("\nRunning segment-level inference (test)...\n")

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    full_path  = Path(row["path"])
    true_class = row["class"]
    video_id   = row["video_id"]

    try:
        pred = clip_model.predict_segment(str(full_path))
        test_results.append({
            "video_id"    : video_id,
            "segment_path": str(full_path),
            "true_class"  : true_class,
            "anomaly_sim" : pred["anomaly_sim"],
            "normal_sim"  : pred["normal_sim"],
            "score"       : pred["score"],
        })
    except Exception as e:
        test_errors.append({"segment": str(full_path), "error": repr(e)})
        print(f"\nERROR: {full_path} -> {repr(e)}")

test_results_df = pd.DataFrame(test_results)
print(f"\nInference done. Success: {len(test_results_df)} | Errors: {len(test_errors)}")
print(test_results_df["true_class"].value_counts())

In [ ]:
test_video_df = aggregate_video_scores(test_results_df)

# Val kalibrasyonundan gelen AYNI oran ve esik test'e uygulanir
# (test'e dokunmuyoruz; TOPK_RATIO ve best_threshold val'de secildi)
test_video_df["score_diff"] = (
    test_video_df[f"topk_a_{TOPK_RATIO}"] - test_video_df[f"topk_n_{TOPK_RATIO}"]
)

test_scores = test_video_df["score_diff"].values
test_y_true = test_video_df["is_anomaly"].astype(int).values
test_y_pred = (test_scores > best_threshold).astype(int)

test_video_df["predicted_is_anomaly"] = test_y_pred.astype(bool)

accuracy     = accuracy_score(test_y_true, test_y_pred)
precision    = precision_score(test_y_true, test_y_pred, zero_division=0)
recall       = recall_score(test_y_true, test_y_pred, zero_division=0)
f1           = f1_score(test_y_true, test_y_pred, zero_division=0)
roc_auc      = roc_auc_score(test_y_true, test_scores)
tn, fp, fn, tp = confusion_matrix(test_y_true, test_y_pred).ravel()
specificity  = tn / (tn + fp + 1e-8)
balanced_acc = (recall + specificity) / 2

print("=" * 50)
print("TEST METRICS")
print("=" * 50)
print(f"Accuracy         : {accuracy:.4f}")
print(f"Balanced Accuracy: {balanced_acc:.4f}")
print(f"Precision        : {precision:.4f}")
print(f"Recall           : {recall:.4f}")
print(f"Specificity      : {specificity:.4f}")
print(f"F1               : {f1:.4f}")
print(f"ROC-AUC          : {roc_auc:.4f}")
print(f"TP={tp}  TN={tn}  FP={fp}  FN={fn}")

# Confusion Matrix
cm = confusion_matrix(test_y_true, test_y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Anomaly'],
            yticklabels=['Normal', 'Anomaly'])
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion Matrix (Test)")
plt.tight_layout()
plt.savefig(f"{PROJECT_ROOT}/outputs/test_confusion_matrix.png", dpi=150)
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(test_y_true, test_scores)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC={roc_auc:.4f}")
plt.plot([0, 1], [0, 1], "k--", alpha=.4)
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title("ROC Curve (Test)"); plt.legend()
plt.tight_layout()
plt.savefig(f"{PROJECT_ROOT}/outputs/test_roc_curve.png", dpi=150)
plt.show()

print("\nPer-video scores:")
print(test_video_df[["video_id", "true_class", "score_diff", "predicted_is_anomaly"]].to_string())

## Save Test Artifacts

In [ ]:
test_video_df.to_csv(f"{PROJECT_ROOT}/outputs/test_video_predictions.csv", index=False)
test_results_df.to_csv(f"{PROJECT_ROOT}/outputs/test_segment_predictions.csv", index=False)

print("✅  Test artifacts saved:")
print(f"    {PROJECT_ROOT}/outputs/test_video_predictions.csv")
print(f"    {PROJECT_ROOT}/outputs/test_segment_predictions.csv")